# Stage 1b – PREDICT Long / Short for any earnings call
Loads the model you trained on **summarized** transcripts (no training) and predicts the next trading day:
**Long** = close above open, **Short** = close at or below open.

**Honest warning:** on 946 real test calls this model was right 52% of the time, about the same as always saying "Short".
Use it to show the pipeline works, not to trade.

**How to run (Kaggle, interactive, no commit needed):**
1. Settings → Accelerator **GPU T4 x2**, Internet **ON**.
2. *Add Input* → datasets `earnings-llm-stage1` (code) and `summary-dataset` (summaries).
3. *Add Input* → your adapter dataset `base-sum-adapter` (the files downloaded from **version 4** of the Base training notebook). Do not add the training notebook itself: its newest version is the zero-shot run, which has no adapter.
4. Set `DATASET` in the configuration cell, then **Run All** once (about 5–10 min to load).
5. Section 4a: one dataset event. Section 4c: many test events at once, with right/wrong. Section 4b: your own transcript. Re-run only the cell you need; each prediction takes a few seconds.


In [ ]:
%%capture
# Official Unsloth install cells (unslothai/notebooks, Llama3 8B) – pinned versions that are known to work together
import os
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle"):
    !pip install pip3-autoremove
    !pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
    !pip install unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
    !pip install transformers==4.57.6
    !pip install --no-deps trl==0.22.2
else:  # Google Colab
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
    !pip install transformers==4.57.6
    !pip install --no-deps trl==0.22.2
!pip install -q scikit-learn

## 1) Configuration

In [ ]:
DATASET     = "base"      # "base" (your adapter) or "full" (friend's adapter) - must match the adapter you added
SPLIT_MODE  = "time"
MAX_SEQ_LEN = 3072
MODEL_NAME  = "unsloth/llama-3-8b-Instruct-bnb-4bit"
TRAIN_RUN   = f"{DATASET}_{SPLIT_MODE}_{MAX_SEQ_LEN}_train_eval_sum"
assert DATASET in ("base", "full"), "DATASET must be base or full"
print(TRAIN_RUN)

## 2) Data, adapter, summarizer

In [ ]:
import os, glob, json, random, time, importlib.util
SEED = 3407
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import numpy as np
def find(*names):
    for name in names:
        hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
        if hits: return hits[0]
    return None
DATA_PATH = find("llm_data_sum.jsonl.gz", "llm_data_sum.jsonl")
CODE_PATH = find("llm_stage1.py")
assert DATA_PATH, "add the summary dataset (llm_data_sum.jsonl) as Input"
assert CODE_PATH, "add the dataset earnings-llm-stage1 (llm_stage1.py) as Input"
spec = importlib.util.spec_from_file_location("llm_stage1", CODE_PATH)
L = importlib.util.module_from_spec(spec); spec.loader.exec_module(L)
records = L.load_records(DATA_PATH)
BY_ID = {r["event_id"]: r for r in records}
print(len(records), "events loaded, e.g.", list(BY_ID)[:5])
hits = sorted(glob.glob(f"/kaggle/input/**/adapter_{TRAIN_RUN}/adapter_config.json", recursive=True))
if not hits:   # adapter uploaded as a dataset with its files at the top level -> use it if it is the only one
    anyc = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
    anyc = [h for h in anyc if "checkpoint" not in h]
    if len(anyc) == 1:
        print("using the only adapter found (check it is the", DATASET, "train_eval_sum one):", os.path.dirname(anyc[0]))
        hits = anyc
if not hits:
    found = sorted({os.path.dirname(h) for h in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)})
    print("adapters found under /kaggle/input:", found or "NONE")
    print("inputs attached:", sorted(os.listdir("/kaggle/input")))
    print("all folders and files under /kaggle/input (first 3 levels):")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root[len("/kaggle/input"):].count(os.sep)
        if depth <= 3:
            print("  " * depth + os.path.basename(root) + "/", files[:8])
assert hits, (f"adapter_{TRAIN_RUN} not found. Add Input -> Datasets -> Your Datasets -> base-sum-adapter (it must contain adapter_config.json "
              "and adapter_model.safetensors, not a zip)")
ADAPTER_DIR = os.path.dirname(hits[0]); print("adapter:", ADAPTER_DIR)

# Repair: files downloaded from the Kaggle web page can get extra lines added at the end
# (e.g. "avatar", "Ask In Chat"), which breaks adapter_config.json. Copy the adapter to
# /kaggle/working and keep only the real content of each text file.
import shutil
FIXED = "/kaggle/working/adapter_fixed"
os.makedirs(FIXED, exist_ok=True)
JUNK = {"avatar", "Ask In Chat"}
for name in os.listdir(ADAPTER_DIR):
    src, dst = os.path.join(ADAPTER_DIR, name), os.path.join(FIXED, name)
    if os.path.isdir(src):
        continue
    if name.endswith(".json"):
        raw = open(src, encoding="utf-8", errors="replace").read()
        obj, end = json.JSONDecoder().raw_decode(raw.lstrip())
        json.dump(obj, open(dst, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
        extra = raw.lstrip()[end:].strip()
        print(f"  {name}: OK" + (f"  (removed {len(extra)} junk characters at the end)" if extra else ""))
    elif name.endswith((".jinja", ".md", ".txt")):
        lines = open(src, encoding="utf-8", errors="replace").read().splitlines()
        while lines and (lines[-1].strip() in JUNK or not lines[-1].strip()):
            lines.pop()
        open(dst, "w", encoding="utf-8").write("\n".join(lines) + "\n")
    else:
        if not os.path.exists(dst):
            os.symlink(src, dst)          # big weights file: link, no copy
assert os.path.exists(os.path.join(FIXED, "adapter_model.safetensors")), \
    "adapter_model.safetensors is missing from your adapter dataset - upload it too"
size_mb = os.path.getsize(os.path.join(FIXED, "adapter_model.safetensors")) / 1e6
print(f"  adapter_model.safetensors: {size_mb:.1f} MB (should be about 167.8 MB)")
cfg = json.load(open(os.path.join(FIXED, "adapter_config.json")))
print("  base model:", cfg.get("base_model_name_or_path"), "| LoRA r:", cfg.get("r"))
ADAPTER_DIR = FIXED
print("using repaired adapter:", ADAPTER_DIR)

In [ ]:
# The same summarizer that built the training data (stage1/2_code/summarize_transcripts.py), copied here unchanged
"""
Stage 1b - extractive summaries of earnings call transcripts for fine-tuning.

Why: the model reads at most 3,072 tokens, so with full transcripts it saw only the first ~12k characters
(about a quarter of a typical call, mostly the CEO's opening remarks). A summary of the key sentences from
the WHOLE call (prepared remarks + Q&A) fits into the same budget without any cutting.

Method (same for train, validation, test and any new call):
  1. Split the call into speaker turns ("Name: text"). Drop all Operator turns.
  2. Find where the Q&A starts. Prepared remarks and Q&A each get half of the character budget
     (unused budget of one part goes to the other).
  3. Split into sentences, drop exact repeats and boilerplate (safe harbor, greetings, "next question" ...).
  4. Score each sentence: finance keywords (guidance, raised/lowered, margin, demand, headwinds ...)
     + numbers / percentages / $ amounts, with a mild preference for short, dense sentences.
  5. Within each part, take the best sentences evenly from its beginning, middle and end
     (round-robin over 3 equal slices), until the budget is used.
  6. Put the chosen sentences back in their ORIGINAL order, under two headers.
Deterministic: the same transcript always gives the same summary. No GPU, no LLM.

Usage:  python summarize_transcripts.py llm_data.jsonl llm_data_sum.jsonl.gz
"""
import gzip
import json
import re
import sys

MAX_CHARS = 8500          # ~2,200 tokens; + instruction & features (<=~470 tokens) stays under 3,072
KEYWORDS = {
    # guidance / outlook
    "guidance": 3, "outlook": 3, "expect": 2, "expects": 2, "expected": 2, "forecast": 3, "anticipate": 2,
    "full-year": 2, "full year": 2, "next quarter": 2, "second half": 2, "fiscal": 1, "target": 1,
    "reaffirm": 3, "reiterate": 2, "raise": 2, "raised": 3, "raising": 3, "lower": 2, "lowered": 3,
    "lowering": 3, "cut": 2, "narrow": 1, "high end": 2, "low end": 2,
    # direction
    "increase": 1, "increased": 1, "decrease": 1, "decreased": 1, "decline": 2, "declined": 2, "grew": 1,
    "growth": 1, "record": 2, "beat": 2, "miss": 2, "missed": 2, "exceeded": 2, "below": 1, "above": 1,
    # tone
    "strong": 1, "strength": 1, "weak": 2, "weakness": 2, "softness": 2, "headwind": 2, "headwinds": 2,
    "tailwind": 2, "tailwinds": 2, "challenging": 2, "pressure": 2, "uncertainty": 2, "uncertain": 2,
    "confident": 2, "momentum": 1, "slowdown": 2, "recovery": 1, "accelerate": 1, "decelerate": 2,
    "inflation": 1, "destocking": 2,
    # financials
    "revenue": 1, "sales": 1, "margin": 2, "margins": 2, "earnings": 1, "eps": 2, "per share": 2,
    "operating income": 1, "cash flow": 1, "demand": 2, "orders": 1, "backlog": 1, "buyback": 1,
    "repurchase": 1, "dividend": 1, "restructuring": 2, "layoffs": 2, "impairment": 2, "pricing": 1,
}
BOILERPLATE = ("forward-looking", "safe harbor", "non-gaap", "reconciliation", "webcast", "replay",
               "press release", "please go ahead", "next question", "your line is open", "thank you for",
               "thanks for taking", "good morning", "good afternoon", "good evening", "welcome to",
               "investor relations", "operator instructions", "sec filings", "risk factors",
               "question-and-answer", "question and answer session", "conference call", "i'll turn",
               "i will turn", "hand it over", "hand the call", "turn the call")
NUM = re.compile(r"(\d+(\.\d+)?\s?%|\$\s?\d|\d+(\.\d+)?\s?(million|billion|basis points|bps))", re.I)
KW_RE = [(re.compile(r"\b" + re.escape(k) + r"\b"), w) for k, w in KEYWORDS.items()]
SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z\"'(“])")
TURN = re.compile(r"^([A-Z][^\n:]{1,60}):\s", re.M)
QA = re.compile(r"(question-and-answer|question and answer|first question|open (it|the (call|line|floor)) "
                r"(up )?(for|to) (your )?questions|operator instructions\].{0,200}question)", re.I | re.S)


def turns(text):
    """[(speaker, text)] in order; text before the first speaker label counts as speaker ''."""
    out, marks = [], list(TURN.finditer(text))
    if not marks or marks[0].start() > 0:
        out.append(("", text[:marks[0].start()] if marks else text))
    for i, m in enumerate(marks):
        end = marks[i + 1].start() if i + 1 < len(marks) else len(text)
        out.append((m.group(1).strip(), text[m.end():end]))
    return out


def qa_start(text):
    """Character position where Q&A begins (first Q&A phrase after the opening); None if not found."""
    for m in QA.finditer(text):
        if m.start() > 3000:
            return m.start()
    return None


def sentences(text):
    return [s.strip() for s in SPLIT.split(re.sub(r"\s+", " ", text)) if s.strip()]


def score(sent):
    low = sent.lower()
    if len(sent.split()) < 8 or any(b in low for b in BOILERPLATE):
        return 0.0
    s = sum(w for rx, w in KW_RE if rx.search(low))
    s += 1.5 * min(len(NUM.findall(sent)), 3)
    return s / (1 + len(sent) / 400)


def pick(sents, budget, n_slices=3):
    """Best sentences evenly from n_slices consecutive slices, within budget; returns indices in order."""
    idx = [i for i in range(len(sents)) if score(sents[i]) > 0]
    slices = [idx[k * len(idx) // n_slices:(k + 1) * len(idx) // n_slices] for k in range(n_slices)]
    ranked = [sorted(s, key=lambda i: (-score(sents[i]), i)) for s in slices]
    chosen, used, k = [], 0, 0
    while any(k < len(r) for r in ranked):
        for r in ranked:
            if k < len(r) and used + len(sents[r[k]]) + 1 <= budget:
                chosen.append(r[k]); used += len(sents[r[k]]) + 1
        k += 1
    return sorted(chosen), used


def summarize(text, max_chars=MAX_CHARS):
    q = qa_start(text)
    if q is None:                                   # no Q&A marker: treat the last 45% as Q&A
        q = int(len(text) * 0.55)
    parts = {"prepared": [], "qa": []}
    pos = 0
    for spk, body in turns(text):
        start = text.find(body, pos); pos = max(pos, start)
        if spk.lower().startswith("operator"):
            continue
        part = "prepared" if start < q else "qa"
        parts[part].extend(sentences(body))
    seen = set()
    for k in parts:                                 # drop exact repeats
        uniq = []
        for s in parts[k]:
            if s not in seen:
                seen.add(s); uniq.append(s)
        parts[k] = uniq
    head_p, head_q = "Prepared remarks (key sentences):\n", "\n\nQuestions and answers (key sentences):\n"
    budget = max_chars - len(head_p) - len(head_q)
    half = budget // 2
    ip, used_p = pick(parts["prepared"], half)
    iq, used_q = pick(parts["qa"], budget - used_p)          # Q&A may use what prepared left over
    if used_q < budget - used_p - 200:                       # Q&A short -> give prepared the rest
        ip, used_p = pick(parts["prepared"], budget - used_q)
    out = head_p + " ".join(parts["prepared"][i] for i in ip)
    if iq:
        out += head_q + " ".join(parts["qa"][i] for i in iq)
    return out


def main(src, dst):
    op = gzip.open if src.endswith(".gz") else open
    n = 0
    with op(src, "rt", encoding="utf-8") as fin, gzip.open(dst, "wt", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip():
                continue
            r = json.loads(line)
            r["transcript_original_chars"] = len(r["transcript"])
            r["transcript"] = summarize(r["transcript"])
            r["summary_chars"] = len(r["transcript"])
            fout.write(json.dumps(r, ensure_ascii=False) + "\n")
            n += 1
    print(f"wrote {n} rows to {dst}")



## 3) Load model + adapter

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# loads the 4-bit base model (named inside adapter_config.json) + your trained LoRA adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")
n_lora = sum(p.numel() for n, p in model.named_parameters() if "lora_" in n)
print("LoRA parameters loaded:", n_lora)
assert n_lora > 0, "adapter was NOT loaded - stop"
print("base model:", json.load(open(f"{ADAPTER_DIR}/adapter_config.json"))["base_model_name_or_path"])
ANSWER_IDS = L.label_token_ids(tokenizer)
print("answer token ids:", ANSWER_IDS)
FastLanguageModel.for_inference(model)
TOKEN_IDS = [ANSWER_IDS["Long"], ANSWER_IDS["Short"]]
print("model ready")

In [ ]:
# ---- fp32 scoring (tested offline: equals full-fp32 scores to ~1e-5, while fp16 scores were off by up to 1/256)
# fp32 scoring: the training run computed the final Long/Short logits in fp16. Logits of size ~16-32 can
# only move in steps of 1/64 in fp16, so P(Long) could only take values 0.5 +- k/256 and ~30% of events
# came out as an exact 0.500 tie. Here the last hidden state is taken from the model and ONLY the two answer
# rows of the output layer are multiplied in float32 -> the model's true, untied P(Long).


def _parts(model):
    base = model.get_base_model() if hasattr(model, "get_base_model") else model   # PeftModel -> LlamaForCausalLM
    return base, base.model, base.get_output_embeddings()


def _answer_rows_fp32(head, token_ids):
    W = head.weight
    if hasattr(W, "quant_state"):                       # only if the output layer itself were 4-bit
        import bitsandbytes.functional as bnbF
        W = bnbF.dequantize_4bit(W.data, W.quant_state)
    return W[token_ids].detach().float()                # [2, hidden]


def _last_hidden_inner(base, inner, head, enc, apply_norm):
    """Strategy A: run the decoder body directly and take the last position's hidden state."""
    out = inner(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], use_cache=False)
    h = out[0][:, -1:, :]
    if apply_norm:
        h = inner.norm(h)
    return h[0, -1]


def _last_hidden_hook(base, inner, head, enc, apply_norm):
    """Strategy B: run the full model as in training-time scoring and capture the input of the output layer."""
    got = {}
    hook = head.register_forward_hook(lambda m, args, out: got.__setitem__("h", args[0]))
    try:
        with torch.inference_mode():
            try:
                base(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], logits_to_keep=1)
            except TypeError:
                base(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
    finally:
        hook.remove()
    if "h" not in got:
        raise RuntimeError("output layer was not called through its module")
    return got["h"][0, -1]


STRATEGIES = {"inner": (_last_hidden_inner, False), "inner+norm": (_last_hidden_inner, True),
              "hook": (_last_hidden_hook, False)}


def score_prompts_fp32(model, tokenizer, prompts, token_ids, strategy="inner", log_every=100):
    base, inner, head = _parts(model)
    W2 = _answer_rows_fp32(head, token_ids).to(base.device if hasattr(base, "device") else "cpu")
    fn, norm = STRATEGIES[strategy]
    model.eval()
    probs, t0 = [], time.time()
    for i, p in enumerate(prompts):
        enc = tokenizer(p, add_special_tokens=False, return_tensors="pt").to(W2.device)
        with torch.inference_mode():
            h = fn(base, inner, head, enc, norm)
            z = W2 @ h.float()                          # [logit_Long, logit_Short] in float32
            probs.append(torch.softmax(z, 0)[0].item())
        if log_every and (i + 1) % log_every == 0:
            el = time.time() - t0
            print(f"  scored {i + 1}/{len(prompts)}  ({el / (i + 1):.2f}s each, "
                  f"~{el / (i + 1) * (len(prompts) - i - 1) / 60:.0f} min left)", flush=True)
    return np.array(probs)


def pick_strategy(model, tokenizer, prompts, token_ids, old_scorer, tol=0.0125):
    """Check on a few prompts that the fp32 score equals the old fp16 score up to fp16 rounding
    (steps of 1/256). Returns the first strategy that agrees; stops the run if none does."""
    p_old = np.asarray(old_scorer(prompts))
    print("old fp16 p_long :", np.round(p_old, 5))
    for name in STRATEGIES:
        try:
            p_new = score_prompts_fp32(model, tokenizer, prompts, token_ids, name, log_every=0)
        except Exception as e:                          # noqa: BLE001 - try the next strategy
            print(f"strategy {name!r} failed: {type(e).__name__}: {e}")
            continue
        diff = float(np.max(np.abs(p_new - p_old)))
        print(f"strategy {name!r}: fp32 p_long {np.round(p_new, 5)}  max diff vs fp16 {diff:.5f}")
        if diff <= tol:
            print(f"-> using {name!r} (agrees with the fp16 scores, only finer)")
            return name
    raise RuntimeError("fp32 scores do not match the original fp16 scores - inspect the scoring log before continuing")


In [ ]:
def predict(instruction, head, transcript, summarize_first=True):
    """Summarizes the transcript like the training data, builds the training prompt, and returns P(Long)."""
    text = summarize(transcript) if summarize_first else transcript
    user, n_tok, cut = L.fit_prompt(tokenizer, instruction, head, text, MAX_SEQ_LEN)
    prompt = L.chat_text(tokenizer, user)
    p = float(score_prompts_fp32(model, tokenizer, [prompt], TOKEN_IDS, "inner", log_every=0)[0])
    print(f"Prediction: {'LONG (close > open)' if p >= 0.5 else 'SHORT (close <= open)'}   P(Long) = {p:.4f}")
    print(f"summary: {len(text):,} characters | prompt tokens: {n_tok}{'  (summary was cut to fit)' if cut else ''}")
    return p

## 4a) Predict an event from the dataset

In [ ]:
# (a) Any event from our dataset (already summarized): change EVENT_ID and re-run this cell
EVENT_ID = "AAPL_2024-05-02"
r = BY_ID[EVENT_ID]
print(EVENT_ID, "| split:", r["split"], "| true label:", r["label"])
p = predict(r["instruction"], r[f"{DATASET}_head"], r["transcript"], summarize_first=False)

## 4c) Check many test events at once

In [ ]:
# (c) Check many test events at once: the model never saw these (Jan-Jun 2024)
N = 20                                     # how many test events to check
test = [r for r in records if r["split"] == "test"]
random.seed(1); sample = random.sample(test, N)
right = 0
for r in sample:
    text = r["transcript"]
    user, n_tok, cut = L.fit_prompt(tokenizer, r["instruction"], r[f"{DATASET}_head"], text, MAX_SEQ_LEN)
    pl = float(score_prompts_fp32(model, tokenizer, [L.chat_text(tokenizer, user)], TOKEN_IDS, "inner", log_every=0)[0])
    pred = "Long" if pl >= 0.5 else "Short"
    right += pred == r["label"]
    print(f"{r['event_id']:<22} P(Long)={pl:.3f}  predicted={pred:<5}  true={r['label']:<5}  {'RIGHT' if pred == r['label'] else 'wrong'}")
print(f"\n{right}/{N} right = {right/N:.0%}   (full test set: 52.3%, always-Short: 51.7%)")


## 4b) Predict your own earnings call

In [ ]:
# (b) Your own earnings call.
#  1. Edit MY_HEAD: keep the same lines and style, change the company, date and numbers.
#     Base = company, date, financial growth. Full also has market performance, analyst grades and EPS surprise.
#  2. Paste the FULL transcript (prepared remarks + Q&A, "Name: text" per speaker) into MY_TRANSCRIPT.
#     It is summarized automatically, exactly like the training data.
TEMPLATE = BY_ID["AAPL_2024-05-02"]
INSTRUCTION = TEMPLATE["instruction"]          # keep as is - the model was trained with this instruction
print("TEMPLATE head (copy and edit):\n" + TEMPLATE[f"{DATASET}_head"])

MY_HEAD = TEMPLATE[f"{DATASET}_head"]           # or paste your edited version here as a """...""" string
MY_TRANSCRIPT = """
Paste the earnings call transcript here (prepared remarks and Q&A).
"""
if len(MY_TRANSCRIPT) < 2000:
    print(">>> No transcript yet: paste the whole call into MY_TRANSCRIPT above, then run only this cell.")
else:
    assert MY_HEAD.endswith("Earnings call transcript:\n"), "MY_HEAD must end with the line 'Earnings call transcript:'"
    p = predict(INSTRUCTION, MY_HEAD, MY_TRANSCRIPT)